# 🎙️ ElevenLabs VoiceLab & Studio (XTTS-v2 Engine)
### Клонирование голоса с сохранением в библиотеку (Voice Library)
---
> * 📚 **VoiceLab Library**: Обучите и сохраните голос один раз — он навсегда останется в вашем выпадающем списке.
> * 🇷🇺 **Нативный русский язык**: Чистое русское произношение без акцента и искажений.
> * 💾 **Google Drive Persistence**: При подключении Google Drive сохраненные голоса никогда не удалятся.
> * 📱 **WebUI для телефона и ПК**: Генерация текста в речь любым выбранным голосом в 1 клик.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка зависимостей и подключение хранилища
Нажмите **Play**. При запросе подключения Google Drive подтвердите доступ, чтобы голоса сохранялись навсегда.

In [ ]:
# @title ⚙️ 1. Установка окружения (Run Setup)
# Автоматически соглашаемся с лицензией Coqui
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# Подключение Google Drive для вечного сохранения вашей библиотеки голосов
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices'
    print('💾 Google Drive подключен! Ваши голоса будут сохраняться навсегда в папку ElevenLabs_Voices.')
except Exception:
    VOICES_DIR = '/content/voices'
    print('💾 Работаем локально (голоса будут сохранены в /content/voices).')

os.makedirs(VOICES_DIR, exist_ok=True)

!nvidia-smi
print('📦 Установка XTTS-v2 и Gradio...')
!pip install -q coqui-tts soundfile gradio
!apt-get install -y -qq ffmpeg
print('✅ Окружение готово!')


## 2. 🚀 Запуск ElevenLabs VoiceLab Studio
Нажмите **Play**. Через ~1 минуту появится ссылка **`Running on public URL: https://xxxx.gradio.live`**.
Перейдите по ней в браузере.

In [ ]:
# @title 🚀 2. Запуск ElevenLabs VoiceLab & Studio (Launch WebUI)
import os
import shutil
import torch
import gradio as gr
import tempfile
from TTS.api import TTS

os.environ['COQUI_TOS_AGREED'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'⚡ Устройство: {device} ({dev_name})')

VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices' if os.path.exists('/content/drive/MyDrive') else '/content/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

print('📥 Загрузка нейросети XTTS-v2...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('🔥 Модель готова к работе!')

def get_saved_voices():
    voices = []
    if os.path.exists(VOICES_DIR):
        for f in sorted(os.listdir(VOICES_DIR)):
            if f.endswith(('.wav', '.mp3', '.ogg')):
                voices.append(os.path.splitext(f)[0])
    return voices if voices else ['[Нет сохраненных голосов]']

def save_new_voice(voice_name, audio_path):
    if not voice_name or not voice_name.strip():
        raise gr.Error('⚠️ Введите название голоса (например: Мой голос, Иван, Диктор)!')
    if not audio_path:
        raise gr.Error('⚠️ Загрузите или запишите аудио-сэмпл голоса (5–15 секунд)!')
    
    clean_name = voice_name.strip().replace('/', '_').replace('\\', '_')
    dest_path = os.path.join(VOICES_DIR, f'{clean_name}.wav')
    shutil.copyfile(audio_path, dest_path)
    
    new_list = get_saved_voices()
    status_msg = f'🎉 Голос "{clean_name}" успешно обучен и сохранен в библиотеку!'
    return (
        gr.update(choices=new_list, value=clean_name),
        gr.update(choices=new_list, value=clean_name),
        status_msg,
        dest_path
    )

def on_voice_select(selected_voice):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        return None
    voice_path = os.path.join(VOICES_DIR, f'{selected_voice}.wav')
    return voice_path if os.path.exists(voice_path) else None

def generate_speech(selected_voice, text, lang, speed):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        raise gr.Error('⚠️ Выберите голос из списка или сначала добавьте новый во вкладке VoiceLab!')
    if not text or not text.strip():
        raise gr.Error('⚠️ Введите текст для озвучки!')
    
    speaker_wav = os.path.join(VOICES_DIR, f'{selected_voice}.wav')
    if not os.path.exists(speaker_wav):
        raise gr.Error(f'⚠️ Файл голоса {selected_voice} не найден!')
    
    out_file = tempfile.mktemp(suffix='.wav')
    tts.tts_to_file(
        text=text.strip(),
        speaker_wav=speaker_wav,
        language=lang,
        speed=float(speed),
        file_path=out_file
    )
    return out_file

custom_css = '''
.gradio-container { font-family: 'Inter', sans-serif; max-width: 1050px !important; margin: auto; }
#btn-generate { background: linear-gradient(135deg, #6366f1 0%, #4f46e5 100%); color: white; font-weight: bold; font-size: 16px; border-radius: 8px; }
#btn-save-voice { background: linear-gradient(135deg, #10b981 0%, #059669 100%); color: white; font-weight: bold; font-size: 15px; border-radius: 8px; }
'''

initial_voices = get_saved_voices()

with gr.Blocks(title='ElevenLabs VoiceLab | XTTS-v2', css=custom_css, theme=gr.themes.Soft(primary_hue='indigo', neutral_hue='slate')) as demo:
    gr.Markdown(
        '''
        # 🎙️ ElevenLabs VoiceLab & Studio (XTTS-v2)
        ### Сохраняйте любые голоса в библиотеку и выбирайте нужный для генерации речи в 1 клик
        '''
    )
    
    with gr.Tabs():
        # --- ВКЛАДКА 1: ОЗВУЧКА СОХРАНЕННЫМ ГОЛОСОМ ---
        with gr.TabItem('⚡ 1. Озвучка текста (Studio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### 👤 Выбор голоса из библиотеки')
                    voice_dropdown = gr.Dropdown(
                        label='Сохраненный голос',
                        choices=initial_voices,
                        value=initial_voices[0],
                        interactive=True
                    )
                    voice_preview = gr.Audio(
                        label='Как звучит этот голос (Оригинал)',
                        type='filepath',
                        interactive=False
                    )
                    voice_dropdown.change(fn=on_voice_select, inputs=[voice_dropdown], outputs=[voice_preview])
                    
                    lang_select = gr.Dropdown(
                        label='Язык озвучки',
                        choices=[('Русский (Russian)', 'ru'), ('English', 'en'), ('Deutsch', 'de'), ('Español', 'es'), ('Français', 'fr')],
                        value='ru'
                    )
                    speed_slider = gr.Slider(label='Скорость речи', minimum=0.7, maximum=1.5, value=1.0, step=0.05)
                    
                with gr.Column(scale=1):
                    gr.Markdown('### 📝 Текст для генерации')
                    text_input = gr.Textbox(
                        label='Введите текст (на русском или другом выбранном языке)',
                        placeholder='Привет! Я говорю твоим сохраненным голосом прямо как в ElevenLabs. Просто выбери меня из списка и введи любой текст!',
                        lines=6
                    )
                    btn_gen = gr.Button('⚡ Озвучить выбранным голосом', elem_id='btn-generate', size='lg')
                    
                    gr.Markdown('### 🔊 Результат')
                    audio_output = gr.Audio(label='Сгенерированное аудио (WAV)', type='filepath')
                    btn_gen.click(
                        fn=generate_speech,
                        inputs=[voice_dropdown, text_input, lang_select, speed_slider],
                        outputs=[audio_output]
                    )

        # --- ВКЛАДКА 2: СОХРАНЕНИЕ / ОБУЧЕНИЕ НОВОГО ГОЛОСА ---
        with gr.TabItem('🎙️ 2. Добавить голос в библиотеку (VoiceLab)'):
            gr.Markdown(
                '''
                ### ➕ Создание нового профиля голоса
                Загрузите сюда образец речи любого человека (или наговорите в микрофон). Голос навсегда сохранится в вашем выпадающем списке!
                '''
            )
            with gr.Row():
                with gr.Column(scale=1):
                    new_voice_name = gr.Textbox(
                        label='Имя голоса (например: Мой голос, Павел Дуров, Диктор 1)',
                        placeholder='Например: Мой голос'
                    )
                    new_voice_audio = gr.Audio(
                        label='Аудио-образец голоса (5–15 секунд без шума и музыки)',
                        type='filepath',
                        sources=['upload', 'microphone']
                    )
                    btn_save = gr.Button('💾 Обучить и сохранить голос в библиотеку', elem_id='btn-save-voice', size='lg')
                    
                with gr.Column(scale=1):
                    save_status = gr.Markdown('ℹ️ Голос еще не добавлен.')
                    saved_preview = gr.Audio(label='Сохраненный эталон', type='filepath', interactive=False)
                    
                    btn_save.click(
                        fn=save_new_voice,
                        inputs=[new_voice_name, new_voice_audio],
                        outputs=[voice_dropdown, voice_dropdown, save_status, saved_preview]
                    )

print('👇 Публичная ссылка для телефона и ПК:')
demo.queue().launch(share=True, debug=False)
